# S1 — Python data structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enriquea/ZebraQ/blob/main/lessons-py/S1_Python_Data_Structures.ipynb)

**ZebraQ — Introduction to Python and bulk RNA-seq data analysis**

By the end of this notebook you will be able to store data in the right Python
structure for the job, index and slice it, and move between NumPy arrays and
pandas DataFrames.

If you have seen the R version of this course, look for the **"In R"** boxes —
they map each Python idiom onto the R one you already know.

## 0. Setup

This cell makes the notebook work both on your own machine and in Google Colab.
You do not need to understand it yet.

In [1]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "pandas", "numpy", "scikit-learn", "session-info"],
        check=True,
    )

import numpy as np
import pandas as pd

print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")

numpy  2.4.6
pandas 2.3.3


## 1. Lists

A **list** is an ordered collection of items. Unlike an R vector, a Python list
can hold items of different types.

> ### ⚠️ The single biggest gotcha coming from R
> **Python counts from 0, R counts from 1.**
> The first element is `v[0]`, not `v[1]`.
> A slice `v[0:5]` includes position 0 up to *but not including* 5.

In [2]:
v = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

print(f"the list        : {v}")
print(f"its type        : {type(v)}")
print(f"how many items  : {len(v)}")
print(f"FIRST element   : {v[0]}      <- v[0], not v[1]")
print(f"LAST element    : {v[-1]}     <- negative indices count from the end")
print(f"first 5 items   : {v[:5]}")
print(f"last 3 items    : {v[-3:]}")
print(f"every 2nd item  : {v[::2]}")

the list        : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
its type        : <class 'list'>
how many items  : 10
FIRST element   : 1      <- v[0], not v[1]
LAST element    : 10     <- negative indices count from the end
first 5 items   : [1, 2, 3, 4, 5]
last 3 items    : [8, 9, 10]
every 2nd item  : [1, 3, 5, 7, 9]


> **In R:** `v <- c(1:10)`, then `v[1]` for the first element, `length(v)` for
> the size, `v[1:5]` for the first five, and `v[length(v)]` for the last.
> Python's `v[-1]` is much shorter than R's `v[length(v)]` — but careful, in R a
> negative index means *drop that element*, which is not what it means here.

Lists are **mutable** — you can change them after creation.

append() - adds item to the end of the list

In [3]:
mixed = [1, "a", True, 3.5]
print(f"mixed types are fine : {mixed}")

mixed.append("new")
print(f"after .append()      : {mixed}")

print(f"is 'a' in the list?  : {'a' in mixed}")

mixed types are fine : [1, 'a', True, 3.5]
after .append()      : [1, 'a', True, 3.5, 'new']
is 'a' in the list?  : True


## 2. Tuples and sets

A **tuple** is an immutable list — once created it cannot be changed. Use it for
things that should not accidentally be modified, like coordinates or a fixed
pair of values.

In [4]:
t = (1, 2, 3)
print(f"tuple      : {t}")
print(f"indexing works the same : {t[0]}")

try:
    t[0] = 99
except TypeError as e:
    print(f"but assignment fails    : {e}")

tuple      : (1, 2, 3)
indexing works the same : 1
but assignment fails    : 'tuple' object does not support item assignment


A **set** is an unordered collection of *unique* items. Very useful in genomics
for comparing gene lists.

In [5]:
genes_a = {"TBX5", "NKX2-5", "GATA4", "TBX5"}   # note the duplicate
genes_b = {"GATA4", "MYH6", "NKX2-5"}

print(f"set drops duplicates : {genes_a}")
print(f"in BOTH  (&)         : {genes_a & genes_b}")
print(f"in EITHER (|)        : {genes_a | genes_b}")
print(f"only in A (-)        : {genes_a - genes_b}")

set drops duplicates : {'TBX5', 'NKX2-5', 'GATA4'}
in BOTH  (&)         : {'NKX2-5', 'GATA4'}
in EITHER (|)        : {'TBX5', 'GATA4', 'MYH6', 'NKX2-5'}
only in A (-)        : {'TBX5'}


> **In R:** sets are just vectors plus `unique()`, `intersect()`, `union()` and
> `setdiff()`. Python makes the set a real type, so `&`, `|` and `-` work directly.

## 3. Dictionaries

A **dictionary** maps keys to values. This is the closest thing to R's *named list*,
and it is one of the most used structures in Python.

In [6]:
d = {"a": [1, 2, 3], "b": "hello", "c": True, "d": 1.5}

print(f"the dictionary   : {d}")
print(f"value under 'a'  : {d['a']}")
print(f"all keys         : {list(d.keys())}")
print(f"all values       : {list(d.values())}")
print(f"is 'b' a key?    : {'b' in d}")

# adding a new entry
d["e"] = 99
print(f"after adding 'e' : {list(d.keys())}")

the dictionary   : {'a': [1, 2, 3], 'b': 'hello', 'c': True, 'd': 1.5}
value under 'a'  : [1, 2, 3]
all keys         : ['a', 'b', 'c', 'd']
all values       : [[1, 2, 3], 'hello', True, 1.5]
is 'b' a key?    : True
after adding 'e' : ['a', 'b', 'c', 'd', 'e']


Looping over a dictionary gives you keys and values together:

In [7]:
for key, value in d.items():
    print(f"  {key} -> {value}")

  a -> [1, 2, 3]
  b -> hello
  c -> True
  d -> 1.5
  e -> 99


> **In R:** `l <- list(a = 1:10, b = "a")`, then `l[["a"]]` to get an element and
> `names(l)` to list the keys. Python's `d["a"]` is the same idea as `l[["a"]]`.
> R's confusing `[` vs `[[` distinction has no equivalent here — `d["a"]` always
> returns the value itself.

## 4. NumPy arrays

For numerical work Python uses **NumPy**. A NumPy array is like an R vector,
matrix or array — the same type in any number of dimensions.

### 4.1 One dimension (an R vector)

In [8]:
a1 = np.array([1, 2, 3, 4, 5])

print(f"array      : {a1}")
print(f"dtype      : {a1.dtype}     <- all elements share ONE type")
print(f"shape      : {a1.shape}")
print(f"a1 * 2     : {a1 * 2}       <- arithmetic applies element-wise")
print(f"a1 + a1    : {a1 + a1}")
print(f"a1.sum()   : {a1.sum()}")
print(f"a1.mean()  : {a1.mean()}")

array      : [1 2 3 4 5]
dtype      : int64     <- all elements share ONE type
shape      : (5,)
a1 * 2     : [ 2  4  6  8 10]       <- arithmetic applies element-wise
a1 + a1    : [ 2  4  6  8 10]
a1.sum()   : 15
a1.mean()  : 3.0


### 4.2 Two dimensions (an R matrix)

> ⚠️ NumPy fills a matrix **row by row** by default. R's `matrix()` fills
> **column by column**. The same numbers therefore land in different places.
> Use `order="F"` if you want R's behaviour.

In [9]:
x = np.arange(1, 13).reshape(3, 4)           # row-wise, the Python default
x_like_r = np.arange(1, 13).reshape(3, 4, order="F")   # column-wise, like R

print("Python default (fills across rows):")
print(x)
print("\nSame call with order='F' (fills down columns, like R's matrix()):")
print(x_like_r)
print(f"\nshape       : {x.shape}   <- (rows, columns), same as R's dim()")
print(f"element [0,0]: {x[0, 0]}")
print(f"first row    : {x[0, :]}")
print(f"first column : {x[:, 0]}")

Python default (fills across rows):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Same call with order='F' (fills down columns, like R's matrix()):
[[ 1  4  7 10]
 [ 2  5  8 11]
 [ 3  6  9 12]]

shape       : (3, 4)   <- (rows, columns), same as R's dim()
element [0,0]: 1
first row    : [1 2 3 4]
first column : [1 5 9]


### 4.3 Matrix operations

Here is the direct translation table for the operations in the R lesson:

| R | Python |
|---|---|
| `x %*% y` | `x @ y` |
| `t(x)` | `x.T` |
| `rowSums(x)` | `x.sum(axis=1)` |
| `colSums(x)` | `x.sum(axis=0)` |
| `rowMeans(x)` | `x.mean(axis=1)` |
| `colMeans(x)` | `x.mean(axis=0)` |

The trick for remembering `axis`: **`axis` is the dimension that disappears.**
`x` has shape (3, 4). `x.sum(axis=1)` collapses the 4 columns, leaving 3 row
totals.

In [10]:
y = np.arange(1, 13).reshape(4, 3)

print("matrix multiplication  x @ y :")
print(x @ y)
print(f"\ntranspose x.T has shape {x.T.shape} (was {x.shape})")
print(f"\nrow sums   x.sum(axis=1) : {x.sum(axis=1)}   <- one per row (3 values)")
print(f"column sums x.sum(axis=0) : {x.sum(axis=0)}   <- one per column (4 values)")
print(f"row means   x.mean(axis=1): {x.mean(axis=1)}")
print(f"col means   x.mean(axis=0): {x.mean(axis=0)}")

matrix multiplication  x @ y :
[[ 70  80  90]
 [158 184 210]
 [246 288 330]]

transpose x.T has shape (4, 3) (was (3, 4))

row sums   x.sum(axis=1) : [10 26 42]   <- one per row (3 values)
column sums x.sum(axis=0) : [15 18 21 24]   <- one per column (4 values)
row means   x.mean(axis=1): [ 2.5  6.5 10.5]
col means   x.mean(axis=0): [5. 6. 7. 8.]


### 4.4 Three dimensions (an R array)

NumPy handles any number of dimensions with the same syntax.

In [11]:
a = np.arange(1, 25).reshape(2, 3, 4)

print(f"shape : {a.shape}   <- 2 layers, each 3 rows x 4 columns")
print(f"\nfirst layer a[0]:\n{a[0]}")
print(f"\nsingle element a[0, 0, 0] : {a[0, 0, 0]}")
print(f"last element   a[-1, -1, -1] : {a[-1, -1, -1]}")
print(f"\nsum over the layers, a.sum(axis=0):\n{a.sum(axis=0)}")

shape : (2, 3, 4)   <- 2 layers, each 3 rows x 4 columns

first layer a[0]:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

single element a[0, 0, 0] : 1
last element   a[-1, -1, -1] : 24

sum over the layers, a.sum(axis=0):
[[14 16 18 20]
 [22 24 26 28]
 [30 32 34 36]]


## 5. pandas Series and DataFrame

**pandas** is where most real data analysis happens. A `DataFrame` is a table
with named columns that can each hold a different type — R's `data.frame`.

In [12]:
s = pd.Series([1.5, 2.5, 3.5], index=["gene1", "gene2", "gene3"], name="expression")
print("A Series is a single labelled column:")
print(s)
print(f"\nlook up by label : {s['gene2']}")

A Series is a single labelled column:
gene1    1.5
gene2    2.5
gene3    3.5
Name: expression, dtype: float64

look up by label : 2.5


In [13]:
df = pd.DataFrame(
    {
        "gene": ["TBX5", "NKX2-5", "GATA4", "MYH6"],
        "expression": [12.5, 8.3, 15.1, 22.0],
        "chromosome": ["12", "5", "8", "14"],
        "significant": [True, False, True, True],
    }
)

print(df)
print(f"\nshape   : {df.shape}        <- like R's dim()")
print(f"columns : {list(df.columns)}")
print("\ndtypes (each column has its own type):")
print(df.dtypes)

     gene  expression chromosome  significant
0    TBX5        12.5         12         True
1  NKX2-5         8.3          5        False
2   GATA4        15.1          8         True
3    MYH6        22.0         14         True

shape   : (4, 4)        <- like R's dim()
columns : ['gene', 'expression', 'chromosome', 'significant']

dtypes (each column has its own type):
gene            object
expression     float64
chromosome      object
significant       bool
dtype: object


The standard first look at any table:

In [14]:
print("df.head(2):")
print(df.head(2))
print("\ndf.describe() — summary of the numeric columns:")
print(df.describe())

df.head(2):
     gene  expression chromosome  significant
0    TBX5        12.5         12         True
1  NKX2-5         8.3          5        False

df.describe() — summary of the numeric columns:
       expression
count    4.000000
mean    14.475000
std      5.745941
min      8.300000
25%     11.450000
50%     13.800000
75%     16.825000
max     22.000000


### 5.1 Selecting rows and columns

> ⚠️ Use `.loc[]` (by **label**) and `.iloc[]` (by **integer position**).
> Being explicit avoids a whole family of confusing bugs.

In [15]:
print(f"one column          :\n{df['gene'].tolist()}")
print(f"\ntwo columns:\n{df[['gene', 'expression']]}")
print(f"\nrow at position 0   :\n{df.iloc[0]}")
print(f"\nboolean filter (expression > 10):\n{df[df['expression'] > 10]}")

one column          :
['TBX5', 'NKX2-5', 'GATA4', 'MYH6']

two columns:
     gene  expression
0    TBX5        12.5
1  NKX2-5         8.3
2   GATA4        15.1
3    MYH6        22.0

row at position 0   :
gene           TBX5
expression     12.5
chromosome       12
significant    True
Name: 0, dtype: object

boolean filter (expression > 10):
    gene  expression chromosome  significant
0   TBX5        12.5         12         True
2  GATA4        15.1          8         True
3   MYH6        22.0         14         True


> **In R:** `df[df$expression > 10, ]` or `subset(df, expression > 10)`.
> The Python version drops the trailing comma — pandas assumes you mean rows.

## 6. Categorical — the equivalent of R's factor

R's `factor` becomes `pd.Categorical`. You need it whenever a column holds a
fixed set of labels and you care about their **order** — for example when you
want chromosomes to plot as 1, 2, … 22, X, Y rather than alphabetically.

In [16]:
fa = pd.Categorical(["a", "b", "c", "a", "b", "c"])
print(f"values     : {list(fa)}")
print(f"categories : {list(fa.categories)}   <- alphabetical by default")

# reorder, exactly like factor(fa, levels = c("c","b","a")) in R
fa2 = fa.reorder_categories(["c", "b", "a"], ordered=True)
print(f"reordered  : {list(fa2.categories)}")
print(f"is ordered : {fa2.ordered}")

values     : ['a', 'b', 'c', 'a', 'b', 'c']
categories : ['a', 'b', 'c']   <- alphabetical by default
reordered  : ['c', 'b', 'a']
is ordered : True


Why the order matters — sorting follows the categories, not the alphabet:

In [17]:
chrom = pd.Categorical(
    ["X", "2", "10", "1", "Y"],
    categories=[str(i) for i in range(1, 23)] + ["X", "Y"],
    ordered=True,
)
print(f"unsorted        : {list(chrom)}")
print(f"sorted properly : {list(chrom.sort_values())}")
print("\nWithout a Categorical, plain text sorting gives the wrong answer:")
print(f"sorted as text  : {sorted(['X', '2', '10', '1', 'Y'])}")

unsorted        : ['X', '2', '10', '1', 'Y']
sorted properly : ['1', '2', '10', 'X', 'Y']

Without a Categorical, plain text sorting gives the wrong answer:
sorted as text  : ['1', '10', '2', 'X', 'Y']


## 7. Putting it together — the iris dataset

The same exercise as the R lesson: filter rows, then group and summarise.

We load iris from `scikit-learn` because it ships with the package and needs no
internet connection.

In [18]:
from sklearn.datasets import load_iris

_iris = load_iris(as_frame=True)
iris = _iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length",
        "sepal width (cm)": "sepal_width",
        "petal length (cm)": "petal_length",
        "petal width (cm)": "petal_width",
    }
)
iris["species"] = pd.Categorical(_iris.target_names[_iris.target])
iris = iris.drop(columns="target")

print(iris.head())
print(f"\nshape: {iris.shape}")

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

shape: (150, 5)


### 7.1 Filtering

Build a boolean mask, then use it to select rows. `&` means *and*, `|` means *or*.

> ⚠️ The parentheses around each condition are **required**. Without them Python
> applies `&` before the comparison and raises an error.

In [19]:
subset = iris[(iris["sepal_length"] > 5) & (iris["sepal_width"] > 3.5)]
print(f"rows matching both conditions: {len(subset)}")
print(subset.head())

# chaining conditions, the equivalent of piping filter() calls in dplyr
setosa_big = iris[
    (iris["sepal_length"] > 5)
    & (iris["sepal_width"] > 3.5)
    & (iris["species"] == "setosa")
]
print(f"\n...and only setosa: {len(setosa_big)} rows")

rows matching both conditions: 16
    sepal_length  sepal_width  petal_length  petal_width species
5            5.4          3.9           1.7          0.4  setosa
10           5.4          3.7           1.5          0.2  setosa
14           5.8          4.0           1.2          0.2  setosa
15           5.7          4.4           1.5          0.4  setosa
16           5.4          3.9           1.3          0.4  setosa

...and only setosa: 13 rows


> **In R (dplyr):**
> ```r
> iris %>% filter(Sepal.Length > 5, Sepal.Width > 3.5, Species == "setosa")
> ```
> pandas has no pipe, so conditions are combined with `&` inside the brackets.

### 7.2 Grouping and summarising

`.groupby()` is pandas' `group_by()` + `summarise()` in one step.

In [20]:
print("mean sepal length per species:")
print(iris.groupby("species", observed=True)["sepal_length"].mean())

print("\nseveral columns at once:")
print(iris.groupby("species", observed=True)[["sepal_length", "sepal_width"]].mean())

print("\nseveral statistics at once:")
print(iris.groupby("species", observed=True)["sepal_length"].agg(["mean", "std", "min", "max"]))

mean sepal length per species:
species
setosa        5.006
versicolor    5.936
virginica     6.588
Name: sepal_length, dtype: float64

several columns at once:
            sepal_length  sepal_width
species                              
setosa             5.006        3.428
versicolor         5.936        2.770
virginica          6.588        2.974

several statistics at once:
             mean       std  min  max
species                              
setosa      5.006  0.352490  4.3  5.8
versicolor  5.936  0.516171  4.9  7.0
virginica   6.588  0.635880  4.9  7.9


> **In R (dplyr):**
> ```r
> iris %>% group_by(Species) %>% summarise(mean(Sepal.Length))
> ```
> The numbers you get here are identical to the R lesson's.

## 8. Exercises

1. Make a list of the numbers 1 to 20. Print every third element.
2. Build a dictionary mapping three gene names to their chromosome. Add a fourth,
   then print all the keys.
3. Create a 4×4 NumPy array containing 1 to 16. Print its transpose, its row sums
   and its column means.
4. Using `iris`, find the mean `petal_length` for each species.
5. How many iris flowers have `petal_width` greater than the overall mean?

*(Hint for 5: build the mask first, then use `.sum()` on it — `True` counts as 1.)*

### Exercise 1 - creating list and printing with slicing

In [21]:
# step 1 - create a list of numbers 1 to 20, 

numbers = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]

print("List of numbers from 1 to 20:", numbers)

print()

## or u can also use range, range (1,21) gives 1 -20

numbers = list(range(1,21))

print("List of numbers 1 to 20:", numbers)


List of numbers from 1 to 20: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

List of numbers 1 to 20: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [22]:
# step 2 - print every third element
# using slicing: [start:stop:start]
# start at index 2 (3rd item), go to end, step by 3..... remember in python indexing starts at 0

every_third = numbers[2::3]
print("Every third element:", every_third)

Every third element: [3, 6, 9, 12, 15, 18]


### Exercise 2 - Building dictionary mapping 3 gene names to their chromosome. Add a fourth, then print all the keys

In [23]:
# Step 1: Create a dictionary with 3 genes
genes = {
    "TBX5": "12",
    "NKX2-5": "5",
    "GATA4": "8"
}

print("Original dictionary:", genes)

# Step 2: Add a 4th gene
genes["MYH6"] = "14"

print("After adding MYH6:", genes)

# Step 3 - Print all keys (gene names)
print("All genes names:", list(genes.keys()))

Original dictionary: {'TBX5': '12', 'NKX2-5': '5', 'GATA4': '8'}
After adding MYH6: {'TBX5': '12', 'NKX2-5': '5', 'GATA4': '8', 'MYH6': '14'}
All genes names: ['TBX5', 'NKX2-5', 'GATA4', 'MYH6']


### Exercise 3 - Create 4x4 NumPy array containing 1 to 16. Print its transpose, its row sum and column means.

In [24]:
import numpy as np

# Step 1: create  4x4 array
array = np.arange(1,17).reshape(4,4)
print("4x4 array:")
print(array)

4x4 array:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]


In [25]:
# Step 2: Print transpose (flip rows and columns)
print("\nTranspose (rows become columns):")
print(array.T)


Transpose (rows become columns):
[[ 1  5  9 13]
 [ 2  6 10 14]
 [ 3  7 11 15]
 [ 4  8 12 16]]


In [26]:
# Step 3: Print row sums (sum each row)
print("\nRow sums (sum of each row):")
print(array.sum(axis=1))  # axis=1 means "sum across columns"



Row sums (sum of each row):
[10 26 42 58]


In [27]:
# Step 4: Print column means (average of each column)
print("\nColumn means (average of each column):")
print(array.mean(axis=0))  # axis=0 means "sum across rows"


Column means (average of each column):
[ 7.  8.  9. 10.]


### Exercise 4 - Using iris, find the mean petal_length for each species.

In [28]:
import pandas as pd

from sklearn.datasets import load_iris

_iris = load_iris(as_frame=True)
iris = _iris.frame.rename (
    columns= {
        "sepal length (cm)": "sepal_length",
        "sepal width (cm)": "sepal_width",
        "petal length (cm)": "petal_length",
        "petal width (cm)": "petal_width"
        }
)

iris["species"] = pd.Categorical(_iris.target_names[_iris.target])
iris = iris.drop(columns="target")


In [29]:
# Step 1: Group by species and calculate mean petal_length, by column
mean_petal_length = iris.groupby("species")["petal_length"].mean()

print("Mean petal length per species:")
print(mean_petal_length)

Mean petal length per species:
species
setosa        1.462
versicolor    4.260
virginica     5.552
Name: petal_length, dtype: float64


/var/folders/td/z21x97312qg8vwcjjxt25_lc0000gp/T/ipykernel_15186/501289444.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_petal_length = iris.groupby("species")["petal_length"].mean()


### Exercise 5 - How many iris flowers have petal_width greater than the overall mean?

In [30]:
# step 1: calculate the overall mean of petal_width
mean_petal_width = iris["petal_width"].mean().round(2)
print(f"Overall mean petal width: {mean_petal_width}")

Overall mean petal width: 1.2


In [31]:
# or
mean_petal_width = iris["petal_width"].mean()
print(f"Overall mean petal width: {mean_petal_width: .2f}")

Overall mean petal width:  1.20


In [32]:
# Step 2: creating a boolean mask, TRue if petal_width > mean
mask = iris ["petal_width"]> mean_petal_width
print("Mask(True = above average):", mask.head())

Mask(True = above average): 0    False
1    False
2    False
3    False
4    False
Name: petal_width, dtype: bool


In [33]:
# Step 3: Count how many True values (i.e., how many flowers are above average)
count = mask.sum()  # True counts as 1, False as 0
print(f"Number of flowers with petal_width > mean: {count}")

Number of flowers with petal_width > mean: 90


## 9. Session information

Recording package versions makes your analysis reproducible — the equivalent of
R's `sessionInfo()`.

In [34]:
import sys

from IPython.display import display

try:
    import session_info
except ModuleNotFoundError:
    # Not an error in your code: this kernel simply does not have session-info.
    # It ships with the course environment (setup/python/environment.yml).
    # The interpreter path below is the fastest way to see which Python the
    # notebook is actually using, which is usually the real problem.
    print("session-info is not installed in this kernel.")
    print("kernel:", sys.executable)
    print("See setup/python/README.md to select the ZebraQ environment.")
else:
    # display() rather than a bare call: inside try/except the returned HTML is
    # no longer the cell's last expression, so Jupyter would not render it.
    display(session_info.show())